# DenseNet-121 흉부 X-Ray 14-질환 분류 — 코드 검증 (100장 테스트)

**목적:** 전체 학습 전에 100장으로 파이프라인이 정상 동작하는지 확인  
**환경:** SageMaker ml.g4dn.xlarge (T4 16GB)  
**확인 항목:**
1. S3에서 이미지 로드 정상 여부
2. Dataset/DataLoader 동작
3. DenseNet-121 forward/backward pass
4. 학습 루프 + validation 동작
5. AUROC 계산 정상 여부

## 1. 환경 설정 및 패키지 임포트

**왜 이 패키지들이 필요한가:**
- `torch`, `torchvision`: PyTorch 딥러닝 프레임워크 + 이미지 처리/사전학습 모델
- `boto3`: AWS S3에서 이미지를 가져오기 위한 AWS SDK
- `PIL (Pillow)`: 이미지 파일을 열고 변환하는 라이브러리
- `sklearn.metrics`: AUROC 등 평가 지표 계산
- `pandas`: CSV 파일 로드/처리

In [ ]:
import os
import io
import time
import json
import random
import numpy as np
import pandas as pd
from PIL import Image
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import boto3
from botocore.config import Config

from sklearn.metrics import roc_auc_score

# ====== 재현성을 위한 시드 고정 ======
# 같은 시드를 쓰면 누가 실행해도 같은 결과가 나옴 (디버깅에 필수)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ====== GPU 확인 ======
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 디바이스: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. 설정값 (Config)

**왜 설정값을 한 곳에 모아두는가:**
- 나중에 하이퍼파라미터를 바꿀 때 여기만 수정하면 됨
- 실험 재현 시 어떤 설정으로 돌렸는지 한눈에 파악 가능

**주요 설정:**
- `TEST_MODE = True`: 100장만 사용하여 코드 검증
- `BATCH_SIZE = 16`: 한 번에 GPU에 올리는 이미지 수 (T4 16GB에 충분)
- `NUM_EPOCHS = 3`: 테스트니까 3에폭만 (본학습 시 30~40)
- `LEARNING_RATE = 1e-4`: Adam 옵티마이저의 학습률

In [ ]:
# ====== 설정값 ======
# 테스트 모드: True면 100장만, False면 전체 데이터
TEST_MODE = True
TEST_SAMPLE_SIZE = 100

# S3 버킷 정보 (본인 작업 버킷 — SageMaker 접근 가능 확인됨)
S3_BUCKET = "pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an"
S3_PREFIX = "data/p10_pa"
REGION = "ap-northeast-2"

# CSV 경로 (SageMaker에 업로드할 경로)
CSV_PATH = "./p10_pa_training.csv"
POS_WEIGHTS_PATH = "./pos_weights.json"

# 14개 질환 라벨 (CheXpert 표준 순서)
LABEL_COLS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity',
    'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia',
    'Pneumothorax', 'Support Devices'
]
NUM_CLASSES = len(LABEL_COLS)  # 14

# 학습 하이퍼파라미터
BATCH_SIZE = 16
NUM_EPOCHS = 3          # 테스트용. 본학습 시 Stage1=10, Stage2=30
LEARNING_RATE = 1e-4    # Adam 학습률
NUM_WORKERS = 2         # DataLoader 병렬 로드 워커 수

# ImageNet 정규화 값 (DenseNet-121이 ImageNet으로 사전학습되었으므로 동일 값 사용)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f"테스트 모드: {TEST_MODE} ({'{}장'.format(TEST_SAMPLE_SIZE) if TEST_MODE else '전체'})")
print(f"S3 버킷: {S3_BUCKET}")
print(f"질환 수: {NUM_CLASSES}개")
print(f"배치 크기: {BATCH_SIZE}")
print(f"에폭 수: {NUM_EPOCHS}")

## 3. CSV 로드 및 데이터 준비

**이 단계에서 하는 일:**
1. CSV에서 이미지 경로 + 14개 질환 라벨 + split 정보 로드
2. 테스트 모드면 100장만 샘플링 (train 70 / val 15 / test 15)
3. pos_weight 로드 (클래스 불균형 보정용)

**pos_weight란?**  
양성이 적은 질환(예: Pleural Other 1.7%)은 모델이 "무조건 음성"이라고 찍어도 정확도가 높음.  
pos_weight로 양성 샘플의 loss를 증폭시켜서, 모델이 희귀 질환도 제대로 학습하게 만듦.

In [ ]:
# ====== CSV 로드 ======
if TEST_MODE:
    # 테스트용: 복사 완료된 PA 이미지 100장 전용 CSV
    df = pd.read_csv("./p10_pa_test100.csv")
    print(f"테스트 모드: 복사 완료된 이미지 100장 로드")
else:
    # 본학습용: 전체 CSV
    df = pd.read_csv(CSV_PATH)
    print(f"전체 데이터: {len(df)}행")

print(f"split 분포:\n{df['split'].value_counts()}\n")

# ====== split별 DataFrame 분리 ======
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'validate'].reset_index(drop=True)
test_df = df[df['split'] == 'test'].reset_index(drop=True)

print(f"최종 데이터: train {len(train_df)} / val {len(val_df)} / test {len(test_df)}")

# ====== pos_weight 로드 ======
with open(POS_WEIGHTS_PATH, 'r') as f:
    pos_weights_dict = json.load(f)

# 라벨 순서에 맞게 tensor로 변환
pos_weight_values = [pos_weights_dict[col] for col in LABEL_COLS]
pos_weights = torch.tensor(pos_weight_values, dtype=torch.float32).to(device)
print(f"\npos_weight 로드 완료 (예: Atelectasis={pos_weight_values[0]:.2f}, Pleural Other={pos_weight_values[10]:.2f})")

## 4. 이미지 Transform 정의

**Transform = 이미지를 모델 입력 형태로 변환하는 파이프라인**

흐름: `S3 JPG → PIL 이미지 → RGB 3채널 → 리사이즈 → (Train만) 랜덤 변형 → 텐서 → 정규화`

**Train vs Val/Test 차이:**
- **Train**: 랜덤 변형(augmentation) 적용 → 모델이 다양한 변형에 강건해짐
  - `RandomResizedCrop(224)`: 랜덤 위치/크기로 224x224 크롭
  - `RandomHorizontalFlip()`: 50% 확률로 좌우 반전
- **Val/Test**: 고정된 변환만 → 일관된 평가를 위해
  - `Resize(256)`: 256으로 리사이즈
  - `CenterCrop(224)`: 정중앙에서 224x224 크롭

**ImageNet 정규화를 쓰는 이유:**  
DenseNet-121이 ImageNet 데이터로 사전학습되었기 때문에, 같은 통계값으로 정규화해야 사전학습 지식을 제대로 활용할 수 있음.

In [ ]:
# ====== Train용 Transform ======
# 랜덤 변형으로 데이터 다양성 확보 → 과적합 방지
train_transform = transforms.Compose([
    transforms.Resize(256),                        # 짧은 변을 256으로
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # 랜덤 크롭 224x224
    transforms.RandomHorizontalFlip(),              # 50% 확률 좌우 반전
    transforms.ColorJitter(brightness=0.1, contrast=0.1),  # 밝기/대비 약간 변형
    transforms.ToTensor(),                          # PIL → Tensor (0~1 범위)
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),  # ImageNet 정규화
])

# ====== Val/Test용 Transform ======
# 고정된 변환으로 일관된 평가
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),                     # 정중앙 224x224 크롭
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Transform 정의 완료")
print(f"  Train: Resize(256) → RandomResizedCrop(224) → HFlip → ColorJitter → Normalize")
print(f"  Eval:  Resize(256) → CenterCrop(224) → Normalize")

## 5. Dataset 클래스 구현

**PyTorch Dataset이란?**  
모델에 데이터를 공급하는 "데이터 창고" 역할. 핵심 메서드 3개:
- `__init__`: 초기화 (CSV 로드, 설정)
- `__len__`: 전체 데이터 개수 반환
- `__getitem__`: 인덱스를 받아 (이미지, 라벨) 한 쌍 반환

**S3 이미지 로드 방식:**  
boto3로 S3에서 이미지 바이트를 다운로드 → PIL로 열기 → RGB 변환 → Transform 적용

**에러 처리:**  
S3에서 이미지 로드 실패 시 검은 이미지(zeros)를 반환 → 학습 중단 방지.  
실제 학습에서는 0-byte 파일이나 깨진 이미지가 있을 수 있으므로 필수.

In [ ]:
class MIMICCXRDataset(Dataset):
    """
    MIMIC-CXR 흉부 X-Ray 데이터셋
    S3에서 이미지를 로드하고, 14개 질환 라벨과 함께 반환
    """
    
    def __init__(self, dataframe, label_cols, s3_bucket, s3_prefix, transform=None, max_retries=3):
        """
        Args:
            dataframe: 이미지 경로 + 라벨이 담긴 DataFrame
            label_cols: 14개 질환 컬럼명 리스트
            s3_bucket: 이미지가 저장된 S3 버킷명
            s3_prefix: 버킷 내 이미지 경로 prefix
            transform: 이미지 변환 파이프라인
            max_retries: S3 로드 실패 시 재시도 횟수
        """
        self.df = dataframe.reset_index(drop=True)
        self.label_cols = label_cols
        self.s3_bucket = s3_bucket
        self.s3_prefix = s3_prefix
        self.transform = transform
        self.max_retries = max_retries
        
        # boto3 클라이언트는 여기서 생성하지 않음 (lazy init)
        # 이유: DataLoader의 num_workers > 0일 때 fork된 프로세스마다 
        # 새 클라이언트를 만들어야 함 (fork safety)
        self._s3_client = None
    
    def _get_s3_client(self):
        """S3 클라이언트 lazy initialization — 매 프로세스마다 새로 생성"""
        if self._s3_client is None:
            self._s3_client = boto3.client(
                's3', 
                region_name=REGION,
                config=Config(
                    max_pool_connections=10,
                    retries={'max_attempts': 3}
                )
            )
        return self._s3_client
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        """
        하나의 샘플 반환: (이미지 텐서, 라벨 텐서)
        
        이미지 텐서: shape (3, 224, 224) — RGB 3채널, 224x224
        라벨 텐서: shape (14,) — 14개 질환 각각 0 또는 1
        """
        row = self.df.iloc[idx]
        
        # ---- 라벨 추출 ----
        labels = torch.tensor(
            [float(row[col]) for col in self.label_cols],
            dtype=torch.float32
        )
        
        # ---- S3에서 이미지 로드 (재시도 포함) ----
        image_path = row['image_path']
        s3_key = f"{self.s3_prefix}/{image_path}"
        
        for attempt in range(self.max_retries):
            try:
                s3 = self._get_s3_client()
                response = s3.get_object(Bucket=self.s3_bucket, Key=s3_key)
                img_bytes = response['Body'].read()
                
                image = Image.open(io.BytesIO(img_bytes)).convert('RGB')
                
                if self.transform:
                    image = self.transform(image)
                
                return image, labels
                
            except Exception as e:
                if attempt < self.max_retries - 1:
                    # SSL 에러 등 네트워크 이슈 시 클라이언트 재생성 후 재시도
                    self._s3_client = None
                    time.sleep(0.5)
                else:
                    print(f"  [WARNING] 이미지 로드 실패 ({self.max_retries}회 시도): {s3_key} — {e}")
                    image = torch.zeros(3, 224, 224)
                    return image, labels

print("MIMICCXRDataset 클래스 정의 완료 (재시도 로직 포함)")

## 6. DataLoader 생성

**DataLoader = Dataset에서 데이터를 배치 단위로 꺼내주는 역할**

- `batch_size=16`: 16장씩 묶어서 GPU에 전달
- `shuffle=True` (Train만): 매 epoch마다 순서를 섞어서 모델이 순서에 의존하지 않게 함
- `num_workers=2`: 2개 프로세스가 병렬로 이미지를 미리 로드 → GPU가 놀지 않게
- `pin_memory=True`: CPU→GPU 데이터 전송 속도 향상 (CUDA 전용)

In [ ]:
# ====== Dataset 생성 ======
train_dataset = MIMICCXRDataset(train_df, LABEL_COLS, S3_BUCKET, S3_PREFIX, transform=train_transform)
val_dataset = MIMICCXRDataset(val_df, LABEL_COLS, S3_BUCKET, S3_PREFIX, transform=eval_transform)
test_dataset = MIMICCXRDataset(test_df, LABEL_COLS, S3_BUCKET, S3_PREFIX, transform=eval_transform)

# ====== DataLoader 생성 ======
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,           # Train은 섞기
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True           # 마지막 불완전 배치 버림 (BatchNorm 안정성)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,           # Val/Test는 섞지 않음
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"DataLoader 생성 완료")
print(f"  Train: {len(train_loader)} 배치 ({len(train_dataset)}장 / batch_size={BATCH_SIZE})")
print(f"  Val:   {len(val_loader)} 배치 ({len(val_dataset)}장)")
print(f"  Test:  {len(test_loader)} 배치 ({len(test_dataset)}장)")

## 7. 데이터 로드 검증

**학습 시작 전에 반드시 확인할 것들:**
1. 이미지 shape이 (3, 224, 224)인지
2. 라벨 shape이 (14,)인지
3. 이미지 값 범위가 정규화 후 적절한지 (대략 -2 ~ 3)
4. S3에서 이미지가 정상 로드되는지

In [ ]:
# ====== 샘플 1개 로드 테스트 ======
print("샘플 1개 로드 테스트...")
sample_img, sample_label = train_dataset[0]
print(f"  이미지 shape: {sample_img.shape}  (기대: torch.Size([3, 224, 224]))")
print(f"  라벨 shape:   {sample_label.shape}  (기대: torch.Size([14]))")
print(f"  이미지 값 범위: [{sample_img.min():.3f}, {sample_img.max():.3f}]")
print(f"  라벨 값: {sample_label.tolist()}")

# ====== 배치 1개 로드 테스트 ======
print("\n배치 1개 로드 테스트...")
start = time.time()
batch_imgs, batch_labels = next(iter(train_loader))
elapsed = time.time() - start
print(f"  배치 이미지 shape: {batch_imgs.shape}  (기대: [{BATCH_SIZE}, 3, 224, 224])")
print(f"  배치 라벨 shape:   {batch_labels.shape}  (기대: [{BATCH_SIZE}, 14])")
print(f"  로드 시간: {elapsed:.2f}초")

# ====== Shape 검증 ======
assert sample_img.shape == (3, 224, 224), f"이미지 shape 오류: {sample_img.shape}"
assert sample_label.shape == (14,), f"라벨 shape 오류: {sample_label.shape}"
print("\n✓ 데이터 로드 검증 통과!")

## 8. DenseNet-121 모델 구성

**DenseNet-121 구조 핵심 개념:**
- ImageNet(자연 이미지 100만장)으로 사전학습된 모델을 가져옴
- 마지막 분류기(classifier)만 교체: 1000클래스(ImageNet) → 14클래스(흉부 질환)
- Dense Connection: 이전 레이어의 출력이 이후 모든 레이어에 직접 연결됨
  → 미세한 특징(음영, 경계)이 깊은 레이어까지 잘 전달됨 → 흉부 X-Ray에 유리

**왜 Sigmoid이고 Softmax가 아닌가?**
- Softmax: 모든 클래스 확률 합 = 1 → 하나만 선택 (multi-class)
- Sigmoid: 각 클래스 독립적으로 0~1 → 여러 질환 동시 가능 (multi-label)
- 한 환자가 폐렴 + 흉수 + 무기폐 동시에 가질 수 있으므로 Sigmoid 사용

**참고:** BCEWithLogitsLoss를 쓰면 내부적으로 Sigmoid를 포함하므로,  
모델 출력에 Sigmoid를 직접 적용하지 않음 (학습 시). 추론 시에만 적용.

In [ ]:
# ====== DenseNet-121 로드 (ImageNet pretrained) ======
# weights='IMAGENET1K_V1': ImageNet으로 사전학습된 가중치 사용
model = models.densenet121(weights='IMAGENET1K_V1')

# ====== classifier 교체 ======
# 원래: Linear(1024, 1000) — ImageNet 1000개 클래스
# 변경: Linear(1024, 14) — 흉부 14개 질환
num_features = model.classifier.in_features  # 1024
model.classifier = nn.Linear(num_features, NUM_CLASSES)  # 1024 → 14

# GPU로 이동
model = model.to(device)

# ====== 모델 구조 요약 ======
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"모델: DenseNet-121")
print(f"전체 파라미터: {total_params:,}개 ({total_params/1e6:.1f}M)")
print(f"학습 가능 파라미터: {trainable_params:,}개")
print(f"분류기: Linear({num_features}, {NUM_CLASSES})")

# ====== Forward pass 테스트 ======
# 더미 입력으로 모델이 정상 동작하는지 확인
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(device)
    output = model(dummy)
    print(f"\nForward pass 테스트:")
    print(f"  입력: {dummy.shape}")
    print(f"  출력: {output.shape}  (기대: [2, 14])")
    print(f"  출력 값 예시: {output[0][:5].cpu().tolist()}  (logits, Sigmoid 적용 전)")
    print(f"\n✓ 모델 구성 및 Forward pass 검증 통과!")

## 9. Loss 함수 + Optimizer 정의

**BCEWithLogitsLoss란?**
- BCE = Binary Cross Entropy (이진 교차 엔트로피)
- WithLogits = 내부적으로 Sigmoid 포함 (수치적으로 더 안정적)
- 14개 질환 각각을 독립적인 이진 분류로 취급
- `pos_weight`: 양성 샘플의 loss를 증폭 → 희귀 질환도 학습하게 함

**Adam Optimizer란?**
- SGD의 개선판: 학습률을 파라미터마다 자동 조절
- 의료 이미지 분야에서 가장 널리 사용되는 옵티마이저
- `lr=1e-4`: 학습률. 너무 크면 발산, 너무 작으면 느림

In [ ]:
# ====== Loss 함수 ======
# pos_weight로 클래스 불균형 보정
# 예: Pleural Other(pos_weight=63.76) → 양성 1개 = 음성 63.76개와 동등한 영향력
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# ====== Optimizer ======
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ====== Learning Rate Scheduler ======
# val loss가 3에폭 연속 개선 안 되면 lr을 0.1배로 줄임
# 학습 후반에 lr을 줄여서 미세 조정하는 효과
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=3, verbose=True
)

print(f"Loss: BCEWithLogitsLoss (pos_weight 적용)")
print(f"Optimizer: Adam (lr={LEARNING_RATE})")
print(f"Scheduler: ReduceLROnPlateau (patience=3, factor=0.1)")

## 10. 학습 + 검증 함수 정의

**학습 루프 (1 epoch) 흐름:**
```
배치 반복:
  1. 이미지, 라벨을 GPU로 이동
  2. Forward pass: 이미지 → 모델 → 예측값 (logits)
  3. Loss 계산: 예측값 vs 실제 라벨
  4. Backward pass: loss로부터 gradient 계산
  5. Optimizer step: gradient 방향으로 파라미터 업데이트
  6. 진행률 출력
```

**검증 루프와의 차이:**
- 학습: `model.train()` → gradient 계산 O, dropout 활성화
- 검증: `model.eval()` + `torch.no_grad()` → gradient 계산 X, dropout 비활성화

**AUROC (Area Under ROC Curve):**
- 모델이 양성/음성을 얼마나 잘 구분하는지 측정 (0.5 = 찍기, 1.0 = 완벽)
- 클래스 불균형에 강건한 지표 → 의료 이미지 분류의 표준 평가 지표

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, epoch, num_epochs):
    """1 에폭 학습"""
    model.train()  # 학습 모드 (dropout, batchnorm 활성화)
    running_loss = 0.0
    total_batches = len(loader)
    epoch_start = time.time()
    
    for batch_idx, (images, labels) in enumerate(loader):
        # 1. GPU로 이동
        images = images.to(device)
        labels = labels.to(device)
        
        # 2. Forward pass
        outputs = model(images)       # shape: (batch_size, 14)
        loss = criterion(outputs, labels)
        
        # 3. Backward pass + 파라미터 업데이트
        optimizer.zero_grad()          # 이전 gradient 초기화
        loss.backward()                # gradient 계산
        optimizer.step()               # 파라미터 업데이트
        
        running_loss += loss.item()
        
        # 4. 진행률 출력
        if (batch_idx + 1) % max(1, total_batches // 5) == 0 or batch_idx == total_batches - 1:
            elapsed = time.time() - epoch_start
            pct = (batch_idx + 1) / total_batches * 100
            avg_loss = running_loss / (batch_idx + 1)
            
            if batch_idx > 0:
                eta = elapsed / (batch_idx + 1) * (total_batches - batch_idx - 1)
                eta_str = f"{eta:.0f}s remaining"
            else:
                eta_str = "계산 중..."
            
            print(f"  [Epoch {epoch+1}/{num_epochs}] {pct:5.1f}% | "
                  f"batch {batch_idx+1}/{total_batches} | "
                  f"loss: {avg_loss:.4f} | "
                  f"{elapsed:.0f}s elapsed | ~{eta_str}")
    
    epoch_loss = running_loss / total_batches
    return epoch_loss


def validate(model, loader, criterion, device):
    """검증/테스트 — loss + AUROC 계산"""
    model.eval()  # 평가 모드 (dropout 비활성화)
    running_loss = 0.0
    all_labels = []
    all_preds = []
    
    with torch.no_grad():  # gradient 계산 안 함 (메모리 절약 + 속도 향상)
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            # Sigmoid 적용하여 확률로 변환 (0~1)
            probs = torch.sigmoid(outputs)
            
            all_labels.append(labels.cpu().numpy())
            all_preds.append(probs.cpu().numpy())
    
    # ====== AUROC 계산 ======
    all_labels = np.concatenate(all_labels, axis=0)  # shape: (N, 14)
    all_preds = np.concatenate(all_preds, axis=0)     # shape: (N, 14)
    
    # 질환별 AUROC 계산
    aurocs = {}
    for i, col in enumerate(LABEL_COLS):
        try:
            # 양성/음성이 둘 다 있어야 AUROC 계산 가능
            if len(np.unique(all_labels[:, i])) > 1:
                aurocs[col] = roc_auc_score(all_labels[:, i], all_preds[:, i])
            else:
                aurocs[col] = float('nan')  # 한쪽만 있으면 계산 불가
        except Exception:
            aurocs[col] = float('nan')
    
    # Mean AUROC (nan 제외)
    valid_aurocs = [v for v in aurocs.values() if not np.isnan(v)]
    mean_auroc = np.mean(valid_aurocs) if valid_aurocs else float('nan')
    
    val_loss = running_loss / len(loader)
    return val_loss, mean_auroc, aurocs


print("학습/검증 함수 정의 완료")

## 11. 학습 실행 (3 에폭 테스트)

**이 셀이 하는 일:**
1. 3 에폭 동안 학습 + 매 에폭 후 검증
2. 에폭별 train loss, val loss, val AUROC 출력
3. best 모델 저장 (val loss 기준)

**100장 테스트이므로 성능은 무의미** — 코드가 에러 없이 돌아가는지만 확인하는 것이 목적

In [ ]:
# ====== 학습 실행 ======
print("=" * 70)
print(f"학습 시작 — {NUM_EPOCHS} 에폭, {'테스트 모드 (100장)' if TEST_MODE else '전체 데이터'}")
print(f"디바이스: {device}")
print("=" * 70)

best_val_loss = float('inf')
train_start = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    
    # ---- Train ----
    print(f"\n{'─'*50}")
    print(f"[Epoch {epoch+1}/{NUM_EPOCHS}] 학습 시작")
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, NUM_EPOCHS)
    
    # ---- Validate ----
    print(f"[Epoch {epoch+1}/{NUM_EPOCHS}] 검증 중...")
    val_loss, val_auroc, val_aurocs = validate(model, val_loader, criterion, device)
    
    # ---- Scheduler step ----
    scheduler.step(val_loss)
    
    # ---- 에폭 결과 출력 ----
    epoch_time = time.time() - epoch_start
    total_elapsed = time.time() - train_start
    remaining = total_elapsed / (epoch + 1) * (NUM_EPOCHS - epoch - 1)
    
    print(f"\n  [Epoch {epoch+1}/{NUM_EPOCHS} 결과]")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val AUROC:  {val_auroc:.4f}")
    print(f"  소요: {epoch_time:.0f}s | 총 경과: {total_elapsed:.0f}s | 남은 예상: ~{remaining:.0f}s")
    print(f"  현재 LR: {optimizer.param_groups[0]['lr']:.2e}")
    
    # ---- Best 모델 저장 ----
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), './best_model_test.pth')
        print(f"  ★ Best 모델 저장! (val_loss: {val_loss:.4f})")

total_time = time.time() - train_start
print(f"\n{'='*70}")
print(f"학습 완료! 총 소요: {total_time:.0f}초 ({total_time/60:.1f}분)")
print(f"Best Val Loss: {best_val_loss:.4f}")
print(f"{'='*70}")

## 12. 테스트셋 최종 평가

**Best 모델을 로드하여 테스트셋에서 최종 성능 측정**
- 질환별 AUROC 출력
- Mean AUROC 출력
- 100장 테스트이므로 수치는 참고용

In [ ]:
# ====== Best 모델 로드 ======
model.load_state_dict(torch.load('./best_model_test.pth', map_location=device))
print("Best 모델 로드 완료\n")

# ====== 테스트셋 평가 ======
test_loss, test_auroc, test_aurocs = validate(model, test_loader, criterion, device)

print(f"{'='*50}")
print(f"테스트셋 최종 결과")
print(f"{'='*50}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Mean AUROC: {test_auroc:.4f}")
print(f"\n{'질환':<35} {'AUROC':>8}")
print(f"{'─'*45}")
for col in LABEL_COLS:
    auc = test_aurocs[col]
    status = f"{auc:.4f}" if not np.isnan(auc) else "N/A (단일 클래스)"
    print(f"{col:<35} {status:>8}")

print(f"{'─'*45}")
print(f"{'Mean AUROC':<35} {test_auroc:>8.4f}")

# ====== 코드 검증 결과 ======
print(f"\n{'='*50}")
print(f"코드 검증 체크리스트")
print(f"{'='*50}")
print(f"  ✓ S3 이미지 로드 정상")
print(f"  ✓ Dataset/DataLoader 동작")
print(f"  ✓ DenseNet-121 Forward/Backward pass 정상")
print(f"  ✓ 학습 루프 (train + validate) 정상")
print(f"  ✓ AUROC 계산 정상")
print(f"  ✓ 모델 저장/로드 정상")
print(f"\n→ TEST_MODE = False로 변경하면 전체 9,118장으로 본학습 가능")

## 13. 정리 (GPU 메모리 해제)

SageMaker 노트북에서 다른 작업 전에 GPU 메모리를 정리합니다.

In [ ]:
# GPU 메모리 정리
import gc
del model, optimizer, criterion, scheduler
del train_loader, val_loader, test_loader
del train_dataset, val_dataset, test_dataset
gc.collect()
torch.cuda.empty_cache()
print("GPU 메모리 정리 완료")
if torch.cuda.is_available():
    print(f"GPU 메모리 사용: {torch.cuda.memory_allocated()/1024**2:.0f}MB / {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f}GB")